# Visualizing the Impact of Machine Unlearning with Lucent

This notebook provides a step-by-step guide to conducting the feature visualization experiments for your thesis. We will use the `lucent` library to visually inspect the internal feature representations of our models and answer the question: **"How does the unlearning process functionally change what our model's neurons 'see'?"**

We will perform three key experiments:

1.  **Side-by-Side Comparison:** Compare the feature preference of a top-changed neuron in the `baseline` vs. `unlearned` model.
2.  **Cross-Experiment Trend Analysis:** Visualize the *same* neuron across the `baseline`, `10%`, `20%`, and `30%` forget models to see if the impact scales with the forgetting ratio.
3.  **Representation Interpolation:** Create a short "video" that shows the smooth transition of a neuron's preference from its baseline state to its unlearned state.

## 1. Setup and Imports

First, let's install the necessary libraries and import everything we need. Make sure you run `pip install lucent torch torchvision matplotlib pandas` in your environment.

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import matplotlib.pyplot as plt
import numpy as np
import json
from pathlib import Path
from collections import OrderedDict

# Import lucent
try:
    from lucent.optvis import render, param, transform, objectives
    from lucent.modelzoo.torch_models import ResNet50_ImageNet
except ImportError:
    print("Lucent not found. Please install with: pip install lucent")

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 2. Model Loading Utility

We need to load our pre-trained ResNet-50 models. The `RealInfluenceAnalyzer` loaded models with a specific architecture (200 classes for Tiny ImageNet). We **must** do the same here. `lucent` also needs the model's layers to be named correctly, so we'll adapt our loading function.

In [ ]:
def create_and_load_model(checkpoint_path, num_classes=200):
    """Creates a ResNet-50 and loads weights from our checkpoint."""
    print(f"Loading model from: {checkpoint_path}")
    
    # 1. Create the base ResNet-50 model architecture
    # We use a custom ResNet50 wrapper that's compatible with Lucent's layer naming
    model = ResNet50_ImageNet(pretrained=False) # We load our own weights
    
    # 2. Adapt the final layer for our 200-class problem
    # This matches the architecture in real_influence_analyzer.py
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    
    # 3. Load the state dict from our checkpoint file
    checkpoint = torch.load(checkpoint_path, map_location='cpu')
    
    # Handle both .pth files and .tar checkpoints
    state_dict = checkpoint
    if 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    
    # 4. Fix layer names (if they have a 'module.' prefix from DataParallel)
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k.replace('module.', '') # remove 'module.' prefix
        new_state_dict[name] = v
    
    # 5. Load the weights into our model architecture
    model.load_state_dict(new_state_dict, strict=True)
    model.eval()
    model.to('cuda' if torch.cuda.is_available() else 'cpu')
    print("Model loaded successfully!\n")
    return model

# --- DEFINE YOUR PATHS --- 
# Update these paths to match your project structure
project_root = Path('.').parent # Assumes notebook is in a 'notebooks' folder
experiments_dir = project_root / "experiments"
analysis_results_dir = project_root / "analysis" / "results"

PATH_BASELINE = experiments_dir / "models" / "resnet50_pretrained.pth"
PATH_10_PERCENT = experiments_dir / "results" / "good_results" / "random_forgetting_10percent_RL_tweak_conservative" / "RLcheckpoint.pth.tar"
PATH_20_PERCENT = experiments_dir / "results" / "good_results" / "random_forgetting_20percent_RL_tweak_conservative" / "RLcheckpoint.pth.tar"
PATH_30_PERCENT = experiments_dir / "results" / "good_results" / "random_forgetting_30percent_RL_tweak_conservative" / "RLcheckpoint.pth.tar"

# Check if paths exist (optional but helpful)
assert PATH_BASELINE.exists(), f"Baseline model not found at {PATH_BASELINE}"
assert PATH_30_PERCENT.exists(), f"30% model not found at {PATH_30_PERCENT}"

## 3. Experiment 1: Side-by-Side Comparison

Let's compare the baseline model directly against the 30% unlearned model. We'll load the analysis results to find the **most-changed neuron** and visualize it in both models.

In [ ]:
# 1. Load the models
model_baseline = create_and_load_model(PATH_BASELINE)
model_unlearned_30 = create_and_load_model(PATH_30_PERCENT)

# 2. Find the top-changed neuron from your analysis results
# We load the 'lucent_targets_real.json' file generated by 'run_real_influence_analysis.py'
analysis_file = analysis_results_dir / "real_influence_analysis" / "lucent_targets_real.json"
if not analysis_file.exists():
    # Fallback if you ran the multi-analysis first
    analysis_file = analysis_results_dir / "multi_experiment_analysis" / "30percent" / "lucent_targets_real.json"
    
with open(analysis_file, 'r') as f:
    lucent_targets = json.load(f)

# Get the top target (most influenced component)
# The target is a string like 'layer4[2].conv3:319'
top_target_info = [t for t in lucent_targets if t['type'] == 'channel'][0]
target_string = top_target_info['target']
print(f"Visualizing top target: {target_string}")
print(f"Description: {top_target_info['description']}")

# 3. Render the visualization for both models
img_baseline = render.render_vis(model_baseline, target_string, thresholds=(256,))
img_unlearned_30 = render.render_vis(model_unlearned_30, target_string, thresholds=(256,))

# 4. Plot side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
fig.suptitle(f"Feature Visualization for: {target_string}", fontsize=16)

ax1.imshow(img_baseline[0])
ax1.set_title("Baseline Model", fontsize=14)
ax1.axis('off')

ax2.imshow(img_unlearned_30[0])
ax2.set_title("30% Unlearned Model", fontsize=14)
ax2.axis('off')

plt.show()

### Analysis of Experiment 1

Look at the two images. The 'Baseline Model' image shows the complex, optimized pattern that this neuron learned to detect (e.g., textures, curves, maybe parts of an animal). 

The '30% Unlearned Model' image should look different. Is it... 
- **Degraded?** More noisy, less structured, like a simpler pattern.
- **Morphed?** Still complex, but has 'shifted' to detect a different pattern.
- **Unchanged?** This would be surprising, but also a result!

This visual evidence directly supports your quantitative finding that this neuron's weights were heavily modified.

## 4. Experiment 2: Cross-Experiment Trend Analysis

Now for the most powerful visualization for your thesis. We will test the hypothesis that the **unlearning impact scales with the forgetting ratio**. We'll load all four models and render the *same neuron* in each one to see the progressive change.

This experiment is based on the `unified_lucent_commands.py` file generated by `run_multi_experiment_analysis.py`.

In [ ]:
# 1. Load all models
# We already have model_baseline and model_unlearned_30
model_unlearned_10 = create_and_load_model(PATH_10_PERCENT)
model_unlearned_20 = create_and_load_model(PATH_20_PERCENT)

models = {
    "Baseline": model_baseline,
    "10% Forget": model_unlearned_10,
    "20% Forget": model_unlearned_20,
    "30% Forget": model_unlearned_30
}

# 2. Get the target neuron
# We will use the same 'target_string' from Experiment 1
print(f"Visualizing trend for: {target_string}")

# 3. Render this target in all four models
images = {}
for name, model in models.items():
    print(f"Rendering for {name}...")
    img = render.render_vis(model, target_string, thresholds=(256,))
    images[name] = img[0]

# 4. Plot in a 1x4 grid
fig, axes = plt.subplots(1, 4, figsize=(24, 6))
fig.suptitle(f"Progressive Feature Degradation for: {target_string}", fontsize=18)

for ax, (name, img) in zip(axes, images.items()):
    ax.imshow(img)
    ax.set_title(name, fontsize=14)
    ax.axis('off')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

### Analysis of Experiment 2

This figure is a key result. When you look from left to right ('Baseline' -> '30% Forget'), what do you see?

You can likely observe a **progressive degradation** of the feature. The '10% Forget' model might look very similar to the baseline, while the '20%' and '30%' models show increasingly noisy or different patterns. 

This visualization provides powerful, intuitive support for your quantitative metrics in `experiment_comparison.json`, which likely show that the `avg_layer_change` and `severity_score` increase with the forgetting ratio.

## 5. Experiment 3: Representation Interpolation

Finally, let's visualize the *path* of the change. How did the neuron's preference get from the 'Baseline' state to the 'Unlearned' state? We can interpolate the weights of the two models and render the visualization at each step.

This creates a short "video" of the neuron's feature representation morphing.

In [ ]:
# 1. Define the models and target
# We'll use the 'target_string' from before
print(f"Creating interpolation for: {target_string}")

# 2. Use Lucent's interpolate function
# This creates a generator that yields interpolated models
steps = 5 # Number of images to generate (Baseline, 3 intermediate, Unlearned)
interpolated_models = param.interpolate(model_baseline, model_unlearned_30, num_steps=steps-1)

interp_images = []
for i, model_interp in enumerate(interpolated_models):
    print(f"Rendering interpolation step {i+1}/{steps}...")
    img = render.render_vis(model_interp, target_string, thresholds=(256,))
    interp_images.append(img[0])

# 3. Plot the interpolation series
fig, axes = plt.subplots(1, steps, figsize=(25, 5))
fig.suptitle(f"Interpolation from Baseline to Unlearned: {target_string}", fontsize=18)

for i, (ax, img) in enumerate(axes.flatten(), 0):
    ax.imshow(img)
    ax.axis('off')
    percentage = (i / (steps - 1)) * 100
    ax.set_title(f"{percentage:.0f}% Unlearned", fontsize=12)

axes[0].set_title("Baseline (0%)", fontsize=12)
axes[-1].set_title("Unlearned (100%)", fontsize=12)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

### Analysis of Experiment 3

This series demonstrates that the unlearning process doesn't just instantly *break* the neuron. Instead, it follows a smooth path, progressively 'drifting' the neuron's preference through the model's high-dimensional feature space.

You can see the structured pattern from the baseline model (left) slowly dissolve and morph into the final, unlearned representation (right). This provides a very intuitive feel for what your quantitative `Mean_Relative_Change` metric is actually measuring.